# TableTrace GPU 보강 실험 (2번 노트북)

**이 노트북은 Kaggle(T4 ×2 권장) 또는 Colab에서 실행합니다. 작성자가 실행해 보지 못했습니다**(개발 컨테이너에 GPU가 없음). CPU에서 확인한 것은 각 스크립트의 작은 무작위 모델 시험뿐입니다(`tests/test_lora_sft_smoke.py`, `tests/test_exaone_grpo_smoke.py`). 처음 도는 설정이므로 실패하면 **그 셀의 출력 전체**를 알려 주세요.

1번 노트북(`gpu_experiments.ipynb`)의 결과는 저장소에 이미 있습니다. 이 노트북은 공고와 대조했을 때 비어 있던 부분을 채우는 실험입니다. **0번 절은 반드시 실행**하고, A~H는 서로 독립이라 골라서 하면 됩니다. 무료 GPU 주 30시간 안에서 권하는 순서는 **H → A → B(7000) → F → G → E → C → A2 → D**입니다. 앞의 것일수록 실패할 일이 적고 결과의 쓸모가 큽니다. A2는 A절의 변수를 쓰므로 A 다음에 실행하세요.

| 절 | 무엇을 재나 | GPU | 예상 시간 | 필요한 것 |
|---|---|---|---|---|
| A | 큰 EXAONE(7.8B, 32B)을 튜닝 없이 쓰면 근거 셀이 맞는가 | 7.8B는 1장, 32B는 2장 | 40분 + 60분 | 없음 |
| A2 | 모델이 도구를 스스로 골라 표를 읽는 방식(Tool Calling) | 1장(32B는 2장) | 30분 + 60분 | A절을 먼저 실행(`TOK` 변수) |
| B | 학습 데이터를 3.6천 → 1만 → 3.4만으로 늘리면 | 1장(2장이면 절반) | 1.5시간, 5시간 | 없음 |
| C | 멀티 GPU 학습(DDP) 속도 | 2장 | 15분 | 없음 |
| D | 강화학습(GRPO)이 근거 셀 오류를 줄이는가 | 1장 | 1.5~2시간 | **본인이 쓴 `training/evidence_reward.py`, `training/grpo.py`** |
| E | 4비트 양자화의 정확도·속도·메모리 | 1장 | 20분 | 1번 노트북 또는 B의 병합 모델 |
| F | 열 번호 형식: 전체 데이터, 시드 2개 더 | 1장 | 1시간 | 없음 |
| G | 영어 화학 문서에서 EXAONE | 1장 | 30분 | 없음 |
| H | 큰 표(공개 dev, 학습에 없던 표 3,072문항): 행 선별 | 1장 | 20분 | 1번 노트북 또는 B의 병합 모델(없으면 새로 학습 30분) |

모든 평가에 `--confidence`를 붙여 모델의 토큰 로그 확률을 함께 저장합니다. 돌려받으면 위험·응답률 곡선(`scripts/risk_coverage.py`)을 EXAONE에 대해서도 그립니다.

## 0. 준비 (필수)

1번 노트북과 같습니다. Kaggle이면 Session options에서 Accelerator = *GPU T4 x2*, Internet = *On*, Add-ons → Secrets에 `HF_TOKEN`.

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
import os, subprocess, time, json, glob
import torch
N_GPU = torch.cuda.device_count()
print("GPU 수:", N_GPU, "| compute capability:", torch.cuda.get_device_capability(0))
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        if not os.environ.get("HF_TOKEN"):
            print("HF_TOKEN이 없습니다. 공개 모델만 받을 수 있습니다.")

REPO = "https://github.com/sokldjs554/tabletrace"
if os.path.isdir("/kaggle/working"):
    ROOT, VENV = "/kaggle/working", "/tmp"
elif os.path.isdir("/content"):
    ROOT = VENV = "/content"
else:
    ROOT = VENV = os.path.join(os.path.expanduser("~"), "work"); os.makedirs(ROOT, exist_ok=True)
if not os.path.isdir(f"{ROOT}/tabletrace"):
    subprocess.run(["git", "clone", "-q", REPO, f"{ROOT}/tabletrace"], check=True)
REPO_DIR = f"{ROOT}/tabletrace"
os.chdir(REPO_DIR)
!git pull -q || echo "git pull 생략"
!pip -q install -e ".[dev]" peft accelerate sentencepiece
!pip -q uninstall -y torchao
!pip -q install uv
if not os.path.isdir(f"{VENV}/venv-vllm"):
    !uv venv --clear {VENV}/venv-vllm -q
    !uv pip install -q --python {VENV}/venv-vllm/bin/python vllm bitsandbytes
print("저장소:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())

In [ ]:
import requests

MODEL = "LGAI-EXAONE/EXAONE-4.0-1.2B"
GPU_NAME = torch.cuda.get_device_name(0).replace(" ", "_")
KO_TRAIN = "data/ko/train.jsonl data/ko/synthetic_train.jsonl"
KO_SETS = (("data/ko/test.jsonl data/ko/synthetic_test.jsonl", "ko_test"), ("data/ko/custom_eval.jsonl", "ko_custom"))

def stop_servers():
    """남아 있는 vLLM 서버는 GPU 메모리와 포트를 쥐고 있다. 다음 서버를 띄우기 전에 모두 끈다."""
    for sig in ("-TERM", "-KILL"):
        for pat in ("vllm serve", "vllm.entrypoints", "VLLM::"):
            subprocess.run(["pkill", sig, "-f", pat], capture_output=True)
        time.sleep(6)
    print(subprocess.run(["nvidia-smi", "--query-gpu=index,memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())

def serve(model, port=8001, num_gpus=1, max_len=2048, dtype="half", extra="", gpu_util=0.88, log=None):
    """vLLM 서버를 띄우고 준비될 때까지 기다린다. 실패하면 로그 마지막 80줄을 보여 준다."""
    stop_servers()
    log = log or f"{ROOT}/vllm_{port}.log"
    e = dict(os.environ, PORT=str(port), PATH=f"{VENV}/venv-vllm/bin:" + os.environ["PATH"], PYTHONPATH="", PYTHONNOUSERSITE="1",
             MODEL=model, NUM_GPUS=str(num_gpus), DTYPE=dtype, MAX_LEN=str(max_len), GPU_UTIL=str(gpu_util), EXTRA_ARGS=extra)
    proc = subprocess.Popen(["bash", "scripts/serving/vllm_tp2.sh"], env=e, stdout=open(log, "w"), stderr=subprocess.STDOUT, cwd=REPO_DIR)
    for _ in range(360):
        try:
            requests.get(f"http://127.0.0.1:{port}/v1/models", timeout=2).raise_for_status()
            return proc
        except Exception:
            if proc.poll() is not None:
                break
            time.sleep(5)
    raise RuntimeError("서버가 뜨지 않았습니다. 서버 로그 마지막 80줄:\n" + "".join(open(log).readlines()[-80:]))

def evaluate(name, tokenizer, sets=KO_SETS, corpus="data/ko/documents.jsonl", max_in=672, table_format="plain", port=8001, concurrency=16, extra=""):
    """같은 검증기·같은 지표로 평가한다. 결과: experiments/eval/<name>_<세트> (검증기 v1)와 <...>_ec (주어 검사 포함 재검증)."""
    for data, suffix in sets:
        out = f"experiments/eval/{name}_{suffix}"
        common = f"--label {GPU_NAME} --tokenizer {tokenizer} --max-in {max_in} --table-format {table_format} --corpus {corpus} --data {data}"
        !python -m tabletrace.eval.run --planner engine --engine-url http://127.0.0.1:{port} --engine-model planner --concurrency {concurrency} --confidence {extra} {common} --out {out} --no-entity-check
        !python -m tabletrace.eval.run --planner engine --replay-from {out} {common} --out {out}_ec

def bench(name, tokenizer, port=8001, table_format="plain"):
    for c in (1, 8, 32):
        !python -m tabletrace.bench.engine_bench --url http://127.0.0.1:{port} --model planner --n 64 --concurrency {c} --tokenizer {tokenizer} --table-format {table_format} --out experiments/bench/engine_{name}_c{c}.json

def train_sft(out, limit=0, extra_n=None, table_format="plain", max_len=768, seed=0, train=KO_TRAIN, corpus="data/ko/documents.jsonl", ddp=False, more=""):
    """LoRA 튜닝. extra_n: 추가 학습 풀(data/ko/pool_train.jsonl.gz)에서 쓸 문항 수(None: 안 씀, 0: 전부). ddp=True면 GPU 2장으로 나눠 학습."""
    pool = "" if extra_n is None else f"--extra-train data/ko/pool_train.jsonl.gz --extra-n {extra_n}"
    corp = corpus + (" data/ko/pool_documents.jsonl.gz" if extra_n is not None else "")
    run = "torchrun --nproc_per_node 2" if ddp else "CUDA_VISIBLE_DEVICES=0 python"
    !{run} scripts/train/exaone_lora_sft.py --model {MODEL} --dtype auto --grad-ckpt --merge --train {train} {pool} --corpus {corp} \
        --epochs 1 --bs 2 --grad-accum 8 --limit {limit} --max-len {max_len} --table-format {table_format} --seed {seed} {more} --out {out}
    t = json.load(open(f"{out}/train_info.json"))
    print("학습", round(t.get("wall_s", 0) / 60, 1), "분 |", t["n_examples"], "예제 |", t["steps"], "step | GPU 프로세스", t.get("world_size", 1))

print("준비 끝. GPU:", GPU_NAME, "x", N_GPU)

## A. 큰 EXAONE을 튜닝 없이 (참고선)

이 프로젝트는 1.2B 모델만 튜닝했습니다. 여기서는 **튜닝하지 않은** EXAONE에 예시 6개(조회 2, 계산 2, 보류 2)를 보여 주고 같은 문법으로 답하게 해서, 같은 검증기로 채점합니다. 볼 것은 두 가지입니다.

- 모델이 26배 커지면(1.2B → 32B) 튜닝 없이도 **근거 셀**이 맞는가.
- 1.2B를 3.6천 문항으로 튜닝한 것(정답 중 근거 오류 1.3%)과 비교해 어느 쪽이 나은가.

순서: 1.2B(기준) → 7.8B AWQ(4비트, T4 한 장) → 32B AWQ(T4 두 장, 텐서 병렬). 32B는 메모리가 빠듯합니다. 서버가 뜨지 않으면 로그를 알려 주고 건너뛰세요.

In [ ]:
SHOTS = f"--few-shot {KO_TRAIN} --shots-per-kind 2"
TOK = MODEL            # 행 예산을 튜닝 모델과 같게 맞추기 위해 1.2B 토크나이저로 센다
REFS = [
    ("ref_1.2b", "LGAI-EXAONE/EXAONE-4.0-1.2B", dict(num_gpus=1, max_len=6144)),
    ("ref_7.8b_awq", "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ", dict(num_gpus=1, max_len=6144, dtype="auto", extra="--trust-remote-code")),
]
if N_GPU >= 2:
    REFS.append(("ref_32b_awq", "LGAI-EXAONE/EXAONE-4.0-32B-AWQ", dict(num_gpus=2, max_len=6144, dtype="auto", gpu_util=0.95)))
for name, model, opts in REFS:
    try:
        proc = serve(model, **opts)
    except RuntimeError as e:
        print("건너뜀:", name, "\n", str(e)[-3000:])
        continue
    !nvidia-smi --query-gpu=index,memory.used --format=csv,noheader
    evaluate(name, TOK, extra=SHOTS, concurrency=8)
    proc.terminate(); proc.wait()

## A2. 모델이 도구를 고르는 방식 (Tool Calling)

지금까지의 에이전트는 순서가 정해져 있습니다. 모델이 프로그램 한 줄을 쓰면 검증기가 도구를 실행합니다. 여기서는 **모델이 표 본문을 보지 않고**, 제목·열 이름·행 수만 받은 뒤 `cell_find`, `row_read`, `column_read`, `calculate` 도구를 스스로 골라 호출하고 `submit` 또는 `abstain`으로 끝냅니다(`src/tabletrace/agent/toolcall.py`). 제출한 프로그램은 같은 검증기를 거칩니다. 표를 도구로 읽으므로 입력 길이 제한이 없습니다.

튜닝하지 않은 EXAONE 4.0으로 시험합니다. vLLM은 도구 호출 해석을 켜고 띄웁니다(`--enable-auto-tool-choice --tool-call-parser hermes`). 서버가 이 옵션을 거부하면 로그를 알려 주세요. A절의 "예시를 보여 주고 한 줄로 답하기"와 같은 문항으로 비교됩니다. 큰 표(H절의 3,072문항 가운데 500개)도 함께 봅니다.

In [ ]:
TOOL_ARGS = "--enable-auto-tool-choice --tool-call-parser hermes"
TOOL_REFS = [("tools_1.2b", "LGAI-EXAONE/EXAONE-4.0-1.2B", dict(num_gpus=1, max_len=8192, extra=TOOL_ARGS))]
if N_GPU >= 2:
    TOOL_REFS.append(("tools_32b_awq", "LGAI-EXAONE/EXAONE-4.0-32B-AWQ", dict(num_gpus=2, max_len=8192, dtype="auto", gpu_util=0.95, extra=TOOL_ARGS)))
for name, model, opts in TOOL_REFS:
    try:
        proc = serve(model, **opts)
    except RuntimeError as e:
        print("건너뜀:", name, "\n", str(e)[-3000:]); continue
    evaluate(name, TOK, extra="--tool-calling", concurrency=4)
    evaluate(name, TOK, sets=(("data/ko/unseen_dev.jsonl", "unseen_dev500"),), corpus="data/ko/unseen_dev_documents.jsonl.gz", extra="--tool-calling --sample 500", concurrency=4)
    proc.terminate(); proc.wait()

## B. 학습 데이터 확대

지금까지의 "전체 데이터" 모델은 조회 2,694문항과 합성 문항으로 학습했습니다. 표 크기 조건을 통과한 표 7,294개 가운데 900개만 썼기 때문입니다. 남은 표 6,111개의 질문 29,907개를 `data/ko/pool_train.jsonl.gz`에 모아 두었습니다(시험·개발 표와 같은 위키 문서에 있는 표는 뺐습니다).

- `EXTRA_N = 7000`: 약 1만 문항, T4 한 장에서 1.5시간 안팎(1번 노트북 실측 0.52초/문항 기준). GPU 두 장이면 `DDP = True`로 절반.
- `EXTRA_N = 0`: 풀 전부(약 3.4만 문항), 한 장 5시간, 두 장 2.5시간. 세션 12시간 제한 안입니다.

먼저 7000을 하고, 시간이 되면 0을 하세요. 결과 이름이 달라 서로 덮어쓰지 않습니다.

In [ ]:
EXTRA_N = 7000          # 0이면 풀 전부
DDP = N_GPU >= 2
POOL_TAG = f"pool{EXTRA_N or 'all'}"
POOL_DIR = f"experiments/exaone_lora_{POOL_TAG}"
train_sft(POOL_DIR, extra_n=EXTRA_N, ddp=DDP)
proc = serve(f"{REPO_DIR}/{POOL_DIR}/merged")
evaluate(f"exaone_{POOL_TAG}_vllm", f"{REPO_DIR}/{POOL_DIR}/merged")
proc.terminate(); proc.wait()

## C. 멀티 GPU 학습 속도 (GPU 2장)

같은 1,000문항을 GPU 한 장과 두 장(DDP, `torchrun --nproc_per_node 2`)으로 학습해 걸린 시간을 비교합니다. 두 장일 때는 프로세스마다 데이터의 절반을 맡고 기울기를 평균합니다. 모델 품질 비교가 아니라 **속도 측정**입니다.

In [ ]:
if N_GPU < 2:
    print("GPU가 1장이라 건너뜁니다.")
else:
    train_sft("experiments/exaone_lora_ddp1", limit=1000, more="--loop builtin")
    train_sft("experiments/exaone_lora_ddp2", limit=1000, ddp=True)
    a, b = (json.load(open(f"experiments/exaone_lora_ddp{k}/train_info.json")) for k in (1, 2))
    print(f"1장 {a['wall_s']:.0f}초 ({a['steps']} step) | 2장 {b['wall_s']:.0f}초 ({b['steps']} step) | 속도 {a['wall_s'] / b['wall_s']:.2f}배")

## D. 강화학습 (GRPO) — 본인이 쓴 보상·손실 코드가 필요합니다

`docs/rewrite/06_training_code.md`의 A, B(`training/evidence_reward.py`, `training/grpo.py`)를 써서 `main`에 병합한 뒤 실행하세요. 없으면 첫 줄에서 안내와 함께 멈춥니다.

- 시작 모델: 300문항으로 튜닝한 모델. 정답 중 근거 셀 오류가 절반을 넘는 모델이라 고칠 것이 많습니다.
- 보상: 검증기가 근거 복구 없이 통과시킨 정답 +1.0, 복구가 있어야 정답 +0.3, 보류 0, 틀린 답 −0.5, 문법 오류 −1.0.
- 한 걸음: 프롬프트 4개 × 샘플 8개. 200걸음에 1~1.5시간 예상(실측 없음).
- 볼 것: 강화학습 전후의 **복구를 끈 정답률**과 **근거 셀 오류율**. 좋아지지 않아도 그대로 기록합니다.

In [ ]:
import importlib.util
if any(importlib.util.find_spec(f"tabletrace.training.{m}") is None for m in ("evidence_reward", "grpo")):
    raise SystemExit("training/evidence_reward.py 와 training/grpo.py 가 없습니다. docs/rewrite/06_training_code.md 를 보고 직접 쓴 뒤 다시 실행하세요.")
START = "experiments/exaone_lora_rlstart"
if not os.path.isfile(f"{START}/merged/config.json"):
    train_sft(START, limit=300)
proc = serve(f"{REPO_DIR}/{START}/merged")
evaluate("exaone_rlstart_vllm", f"{REPO_DIR}/{START}/merged")          # 강화학습 전
proc.terminate(); proc.wait(); stop_servers()

In [ ]:
GRPO_DIR = "experiments/exaone_grpo"      # 메모리가 모자라면 --group 4 --chunk 1 로 줄이세요
!CUDA_VISIBLE_DEVICES=0 python scripts/train/exaone_grpo.py --model {MODEL} --adapter {START} --dtype auto \
    --train {KO_TRAIN} --dev data/ko/dev.jsonl data/ko/synthetic_dev.jsonl --corpus data/ko/documents.jsonl \
    --steps 200 --prompts 4 --group 8 --chunk 2 --lr 2e-5 --eval-every 25 --dev-n 64 --merge --out {GRPO_DIR}
t = json.load(open(f"{GRPO_DIR}/train_info.json"))
print("갱신", t["updates_applied"], "/", t["steps"], "| 고른 걸음", t["kept_step"], "| 개발셋 기록", [(h["step"], h["mean_reward"], h["correct"]) for h in t["dev_history"]])
proc = serve(f"{REPO_DIR}/{GRPO_DIR}/merged")
evaluate("exaone_grpo_vllm", f"{REPO_DIR}/{GRPO_DIR}/merged")           # 강화학습 후
proc.terminate(); proc.wait()

## E. 4비트 양자화 (bitsandbytes)

튜닝한 모델을 vLLM이 4비트로 올리게 해서(`--quantization bitsandbytes`), fp16과 같은 문항으로 정확도·처리량·GPU 메모리를 비교합니다. `QMODEL`은 1번 노트북의 전체 데이터 모델이 이 세션에 없으면 B에서 만든 모델을 씁니다. 둘 다 없으면 300문항 모델을 새로 만듭니다.

In [ ]:
cands = [f"{REPO_DIR}/experiments/exaone_lora_full/merged"] + sorted(glob.glob(f"{REPO_DIR}/experiments/exaone_lora_pool*/merged")) + [f"{REPO_DIR}/experiments/exaone_lora_rlstart/merged"]
QMODEL = next((c for c in cands if os.path.isfile(f"{c}/config.json")), None)
if QMODEL is None:
    train_sft("experiments/exaone_lora_rlstart", limit=300); QMODEL = f"{REPO_DIR}/experiments/exaone_lora_rlstart/merged"
QTAG = QMODEL.split("/")[-2].replace("exaone_lora_", "")
print("양자화할 모델:", QMODEL)
for name, extra in ((f"q_fp16_{QTAG}", ""), (f"q_bnb4_{QTAG}", "--quantization bitsandbytes")):
    try:
        proc = serve(QMODEL, extra=extra, dtype="half" if not extra else "auto")
    except RuntimeError as e:
        print("건너뜀:", name, "\n", str(e)[-3000:]); continue
    mem = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split()
    json.dump({"model": QMODEL, "vllm_args": extra, "gpu_memory_used_mib": [int(x) for x in mem], "note": "vLLM은 gpu-memory-utilization만큼 미리 잡으므로 이 값은 모델 크기가 아니라 서버가 잡은 양이다"},
              open(f"experiments/bench/engine_{name}_memory.json", "w"), indent=1)
    evaluate(f"exaone_{name}_vllm", QMODEL)
    bench(name, QMODEL)
    proc.terminate(); proc.wait()

## F. 열 번호 형식 — 전체 데이터와 시드 추가

1번 노트북에서 300문항·시드 하나로 본 결과(근거 셀 오류 53~65% → 3.2%)를 넓힙니다. (1) 300문항을 시드 1, 2로 두 번 더, (2) 전체 데이터에서 한 번. 전체 데이터의 `plain` 근거 오류는 이미 1.3%라 차이가 작을 것으로 예상합니다. 그 예상이 맞는지 보는 실험입니다.

In [ ]:
for seed in (1, 2):
    d = f"experiments/exaone_lora_colid_s{seed}"
    train_sft(d, limit=300, table_format="colid", max_len=896, seed=seed)
    proc = serve(f"{REPO_DIR}/{d}/merged")
    evaluate(f"exaone_colid_s{seed}_vllm", f"{REPO_DIR}/{d}/merged", max_in=800, table_format="colid")
    proc.terminate(); proc.wait()
d = "experiments/exaone_lora_full_colid"
train_sft(d, table_format="colid", max_len=896)
proc = serve(f"{REPO_DIR}/{d}/merged")
evaluate("exaone_full_colid_vllm", f"{REPO_DIR}/{d}/merged", max_in=800, table_format="colid")
proc.terminate(); proc.wait()

## G. 영어 화학 문서

한국어 표와 화학 합성 문항(820개)을 함께 학습한 EXAONE을 화학 평가 세 세트로 채점합니다. 교과서 표의 수작업 32문항과 합성 개발 80문항, 그리고 **실제 화학 논문 12편(arXiv, CC BY 4.0)의 표 20개에 대한 수작업 62문항**(`data/chem_papers/`)입니다. 논문 표는 학습에 없던 것이고, 단위 행·수식 기호·불확도 표기가 섞여 있어 더 어렵습니다. mT5-small은 여기서 약했습니다(교과서 수작업 18.8%, 논문 표 19~23%, 휴리스틱은 43.5%). 같은 모델이 한국어 성능을 유지하는지도 함께 봅니다.

In [ ]:
CHEM_DIR = "experiments/exaone_lora_kochem"
train_sft(CHEM_DIR, train=KO_TRAIN + " data/chem/synthetic_train.jsonl", corpus="data/ko/documents.jsonl data/chem/documents.jsonl")
proc = serve(f"{REPO_DIR}/{CHEM_DIR}/merged")
evaluate("exaone_kochem_vllm", f"{REPO_DIR}/{CHEM_DIR}/merged", corpus="data/chem/documents.jsonl",
         sets=(("data/chem/custom_eval.jsonl", "chem_custom"), ("data/chem/synthetic_dev.jsonl", "chem_syndev")))
evaluate("exaone_kochem_vllm", f"{REPO_DIR}/{CHEM_DIR}/merged", corpus="data/chem_papers/documents.jsonl",
         sets=(("data/chem_papers/custom_eval.jsonl", "chem_papers"),), extra="--row-select overlap")      # 실제 화학 논문 12편의 표 62문항
evaluate("exaone_kochem_vllm", f"{REPO_DIR}/{CHEM_DIR}/merged")       # 한국어 성능이 유지되는지
proc.terminate(); proc.wait()

## H. 큰 표와 공개 dev 분할

지금까지의 시험 표는 16행·7열·900자 이하였습니다. `data/ko/unseen_dev.jsonl`은 **공개 dev 분할에서 학습에 쓰이지 않은 표의 질문 3,072개**이고 97%가 그 한도를 넘는 표입니다. 표가 입력에 다 들어가지 않을 때 앞에서부터 자르는 방식(`first`)과 질문과 겹치는 행을 고르는 방식(`overlap`, 행 번호는 원래 값 유지)을 비교합니다. 모델은 다시 학습하지 않고, 이 세션에 있는 튜닝 모델을 그대로 씁니다. 입력 예산을 1,500토큰으로 늘린 경우도 함께 봅니다(학습 때 본 길이는 672 이하).

In [ ]:
cands = [f"{REPO_DIR}/experiments/exaone_lora_full/merged"] + sorted(glob.glob(f"{REPO_DIR}/experiments/exaone_lora_pool*/merged")) + [f"{REPO_DIR}/experiments/exaone_lora_rlstart/merged"]
LMODEL = next((c for c in cands if os.path.isfile(f"{c}/config.json")), None)
if LMODEL is None:
    train_sft("experiments/exaone_lora_full"); LMODEL = f"{REPO_DIR}/experiments/exaone_lora_full/merged"
LTAG = LMODEL.split("/")[-2].replace("exaone_lora_", "")
print("평가할 모델:", LMODEL)
proc = serve(LMODEL, max_len=4096)
UNSEEN = (("data/ko/unseen_dev.jsonl", "unseen_dev"),)
for rs, max_in in (("first", 672), ("overlap", 672), ("first", 1500)):
    evaluate(f"exaone_{LTAG}_{rs}{max_in}_vllm", LMODEL, sets=UNSEEN, corpus="data/ko/unseen_dev_documents.jsonl.gz", max_in=max_in, extra=f"--row-select {rs}")
proc.terminate(); proc.wait()

## I. 결과 돌려주기

아래 셀이 이 노트북에서 만든 결과만 묶습니다(모델 가중치는 담지 않습니다). `gpu_results_2.tgz`를 받아서 대화에 올려 주세요.

In [ ]:
stop_servers()
json.dump({"notebook": "gpu_experiments_2", "platform": "kaggle" if os.path.isdir("/kaggle/working") else "colab",
           "gpus": [torch.cuda.get_device_name(i) for i in range(N_GPU)], "commit": subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip(),
           "finished": time.strftime("%Y-%m-%dT%H:%M:%S")}, open("experiments/run_info_2.json", "w"), indent=1)
pats = ("experiments/eval/ref_*", "experiments/eval/tools_*", "experiments/eval/exaone_pool*", "experiments/eval/exaone_rlstart_*", "experiments/eval/exaone_grpo_*", "experiments/eval/exaone_q_*",
        "experiments/eval/exaone_colid_s*", "experiments/eval/exaone_full_colid_*", "experiments/eval/exaone_kochem_*", "experiments/eval/exaone_*_unseen_dev*", "experiments/bench/engine_q_*",
        "experiments/exaone_lora_*/train_info.json", "experiments/exaone_grpo/train_info.json", "experiments/exaone_grpo/train_log.jsonl", "experiments/run_info_2.json")
paths = [p for pat in pats for p in sorted(glob.glob(pat))]
print("담는 것:", len(paths), "개")
out = f"{ROOT}/gpu_results_2.tgz"
r = subprocess.run(["tar", "czf", out, *paths], capture_output=True, text=True)
print(r.stderr or "tar ok", "|", out, os.path.getsize(out), "bytes")